## Customer Segementation (RFM Analysis)
## `Author: Nishan Karki`
<hr>

## Table of Contents

## [1. Setup & Configuration](#setup-configuration)
- [**1.1 Import Libraries**](#import-libraries)
- [**1.2 Verify Tables**](#verify-tables)


##  [2. Business Context](#business-context)
   - [**2.1 Objective**](#objective)
   - [**2.2 Data Scope Definition**](#data-scope)

## [3. Analysis & Metrics](#analysis-metrics)
   - [**3.1 Recency Analysis**](#recency-analysis)
   >##### [Summary](#recency-analysis-summary)
   
   - [**3.2 Frequency Analysis**](#frequency-analysis)
   >##### [Summary](#frequency-analysis-summary)
   
   - [**3.3 Monetary Analysis**](#monetary-analysis)
   >##### [Summary](#monetary-analysis-summary)
   

## [4. Key Findings & Business Interpretation](#key-findings--business-interpretation)



<a id='setup-configuration'></a>
# 1. Setup & Configuration:

<a id='import-libraries'></a>
## 1.1 Import  Libraries:

In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine
import os
import numpy as np
import sys
from IPython.display import display, Markdown, Image
from pathlib import Path

import chart_studio.plotly as py
import plotly.express as px
import plotly.graph_objects as go
import cufflinks as cf
import matplotlib.pyplot as plt
%matplotlib inline

import math
import warnings
warnings.filterwarnings("ignore")
import importlib
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=True)
cf.go_offline()

# Add the project root to the system path:
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import custome modules:
from src.rfm_analysis import rfm_analysis_metrics, rfm_analysis_summary
from plots import rfm_analysis_plots
from utils import helper

In [2]:
# Create a connection to the ecommerce database which is required for RFM metric function calls:
conn = sqlite3.connect("../data/ecommerce.db")

In [3]:
# A helper function to reload the package to pick up necessary changes:
def reload_package():
    importlib.reload(rfm_analysis_metrics)
    importlib.reload(rfm_analysis_summary)
    importlib.reload(rfm_analysis_plots)
    importlib.reload(helper)
    print("Packages reloaded!!")

<a id='verify-tables'></a>
## 1.2 Verify Tables:

In [4]:
# Create a global path for all SQL files that is related to the RFM analysis:
SQL_BASE_PATH = Path("../sql/03_Customer_Segmentation_(RFM_Analysis)")

In [5]:
# Get the list of the available tables:
tables = helper.extract_table_names(conn=conn)
print(f"The list of the tables:\n{tables}")

The list of the tables:
                           name
0                     customers
1                   geolocation
2                        orders
3                   order_items
4                order_payments
5                 order_reviews
6                      products
7                       sellers
8  product_category_translation


<a id='business-context'></a>
# 2. Business Context:

<a id='objective'></a>
## 2.1 Objective:
To segment customers based on `Recency, Frequency, and Monetary` value in order to identify high-value, loyal, at-risk, and churn-prone customer groups, enabling targeted retention strategies, personalized marketing, and revenue optimization.

<a id='data-scope'></a>
## 2.2 Data Scope Definition:

For RFM analysis, the orders with `order_status=delivered` are considered. 
This ensures that:
1. Revenue has been fully realized
2. The transaction represents a completed customer purchase
3. Customer behavior is measured based on confirmed buying activity rather than intent or failed transactions

**NOTE:** A temporary table `customer_order_info`is created for further analysis:

In [6]:
rfm_analysis_metrics.create_temp_table_customer_order_info(conn=conn, sql_base_path=SQL_BASE_PATH)

Temporary table `customer_order_info` successfully created !!


<a id='analysis-metrics'></a>
# 3. Analysis & Metrics:

<a id='recency-analysis'></a>
## 3.1  Recency Analysis:

- Recency asks **"When did the customer made their last purchase that was successfull or the `order status was delivered`?**
- In business terms:
    - `Low Recency (recent purchases)`: active or engaged customers
    - `High Recency (long-time ago)`: inactive, churn-risk customers
    
But this dataset is old and using the current date won't make sense, so for this analysis, the latest date from the 
`order_purchase_timestamp + 10 DAYS` will be used as the threshold.

### 3.1.1 Execute the `get_recency_analysis()` function from the `rfm_analysis_metrics` module:

In [7]:
recency_table_full = rfm_analysis_metrics.get_recency_analysis(conn=conn, sql_base_path=SQL_BASE_PATH)
recency_table_full.head(3)

,customer_id,latest_purchase_timestamp,date_threshold,days_since_last_purchase,bins
0,00012a2ce6f8dcda20d059ce98491703,2017-11-14,2018-09-08,297,50
1,000161a058600d5901f007fab4c27140,2017-07-16,2018-09-08,418,80
2,0001fd6190edaaf884bcaf3d49edf079,2017-02-28,2018-09-08,556,110


### 3.1.2 Now, we will use the **bins** feature of the `recency_table_full` table to group the customers and count the unique IDs:

In [8]:
customers_per_bins = recency_table_full.groupby(['bins'])['customer_id'].nunique()

In [9]:
customers_per_bins

bins
0       9196
10      9545
20     11213
30     11553
40     11276
50     11198
60      7076
70      6895
80      5844
90      5591
100     3960
110     2747
120      118
130       90
140      176
Name: customer_id, dtype: int64

### 3.1.3 Mapping the **bins** (`a 50-day recency bins`) values to days range for the visualization purpose:

In [10]:
# Backup if something goes wrong:
customers_per_bins_labeled = customers_per_bins.copy()

# Mapping dictionary:
bin_mapping = {
    0: '0-50',
    10: '50-100',
    20: '100-150',
    30: '150-200',
    40: '200-250',
    50: '250-300',
    60: '300-350',
    70: '350-400',
    80: '400-450',
    90: '450-500',
    100: '500-550',
    110: '550-600',
    120: '600-650',
    140: '650+'
}

customers_per_bins_labeled.index = customers_per_bins_labeled.index.map(bin_mapping)
customers_per_bins_labeled

customers_per_bins_labeled_df = customers_per_bins_labeled.to_frame().reset_index().rename(columns={'bins': 'Days Interval',
                                                                                                   'customer_id': 'Total Customers'})

This is how a well-formatted results looks like for the customer counts by the bin size where we use **a 50-day recency bins interval**:

In [11]:
customers_per_bins_labeled_df

,Days Interval,Total Customers
0,0-50,9196
1,50-100,9545
2,100-150,11213
3,150-200,11553
4,200-250,11276
5,250-300,11198
6,300-350,7076
7,350-400,6895
8,400-450,5844
9,450-500,5591


### 3.1.4 Call the `plot_customer_distribution_recency_bar_chart()` function from `rfm_analysis_plots` module:

In [12]:
fig = rfm_analysis_plots.plot_customer_distribution_recency_bar_chart(df = customers_per_bins_labeled_df)

<a id='recency-analysis-summary'></a>
## Summary:

### Call the `revenue_analysis_summary()` function from `rfm_analysis_summary` module to <br> <br> summarize the `Recency Analysis` section:

In [13]:
reload_package()

Packages reloaded!!


In [14]:
rfm_analysis_summary.get_revenue_analysis_summary(recency_table_full, customers_per_bins_labeled_df, fig)


## Recency Analysis Conclusion:

Recency measures how long it has been since a customer’s **most recent completed purchase.** 

In this analysis:
- Only orders with `delivered` status were considered to ensure revenue realization.
- **Recency** was calculated as the number of days between a customer's latest purchase timestamp and a threshold set as
`DATE(MAX(order_purchase_timestamp), '+10 days')` .i.e. **2018-09-08**. This approach was 
chosen because the dataset is historical, and using the current date would distort recency values.

- Each customer is recorded once using a `WINDOW FUNCTION` to capture their most recent purchase.

The resulting table `recency_table_full` holds the entire information at a customer level, leaving space for further
segmentation and drill-down analysis when required, `recency_table_full` holds a data of **96478** rows 
and **5** features, as a glimpse, here is what it looks like: 

,customer_id,latest_purchase_timestamp,date_threshold,days_since_last_purchase,bins
0,00012a2ce6f8dcda20d059ce98491703,2017-11-14,2018-09-08,297,50
1,000161a058600d5901f007fab4c27140,2017-07-16,2018-09-08,418,80
2,0001fd6190edaaf884bcaf3d49edf079,2017-02-28,2018-09-08,556,110
3,0002414f95344307404f0ace7a26f1d5,2017-08-16,2018-09-08,387,70
4,000379cdec625522490c315e70c7a9fb,2018-04-02,2018-09-08,158,30



To improve interpretability, customers were grouped into **50-day recency bins**, allowing the analysis to be performed
at an aggregated level while preserving the underlying customer-level data:


,Days Interval,Total Customers
0,0-50,9196
1,50-100,9545
2,100-150,11213
3,150-200,11553
4,200-250,11276
5,250-300,11198
6,300-350,7076
7,350-400,6895
8,400-450,5844
9,450-500,5591



The chart below visualizes the distribution of customers across recency bins, highlighting how customer activity
declines as time since last purchase increases:


<a id='frequency-analysis'></a>
## 3.2 Frequency Analysis:

- Frequency asks **"How often does a customer make a purchase?"**
- In business terms,
    - `High Frequency`: loyal customers
    - `Low Frequency`: one-time customer or chustomer who are at churn-risk


### 3.2.1 Call the `get_customer_order_frequency_analysis()` function from the `rfm_analysis_metrics` module:

In [15]:
customer_order_frequency_interpretation = rfm_analysis_metrics.get_customer_order_frequency_analysis(conn=conn,
                                                                                                    sql_base_path=SQL_BASE_PATH)

In [16]:
# The returned dataframe consists of the information below:
customer_order_frequency_interpretation.head()

,customer_unique_id,total_orders,Interpretation
0,0000366f3b9a7992bf8c76cfdf3221e2,1,One-Time Buyer
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,One-Time Buyer
2,0000f46a3911fa3c0805444483337064,1,One-Time Buyer
3,0000f6ccb0745a6a4b88665a16c9f078,1,One-Time Buyer
4,0004aac84e0df4da2b147fca70cf8255,1,One-Time Buyer


### 3.2.2 Use the `Interpretation` feature to count the customer and their contribution in % out of all the total orders made:

In [17]:
customer_order_frequency_interpretation_counts = customer_order_frequency_interpretation['Interpretation'].value_counts()
customer_order_frequency_interpretation_counts = customer_order_frequency_interpretation_counts.to_frame().reset_index()
customer_order_frequency_interpretation_counts.rename(columns={'count':'Total Counts'}, inplace=True)
customer_order_frequency_interpretation_counts['Customer Share (%)'] = np.round((customer_order_frequency_interpretation_counts['Total Counts']
                                                                        / customer_order_frequency_interpretation_counts['Total Counts'].sum()) * 100, 3)

In [18]:
customer_order_frequency_interpretation_counts

,Interpretation,Total Counts,Customer Share (%)
0,One-Time Buyer,90557,97.000
1,Returning Customer,2573,2.756
2,Loyal Customer,181,0.194
3,Very Loyal,28,0.030
4,VIP Customers,19,0.020


### 3.2.3 Call the `plot_customer_purchase_frequency_bar_chart()` function from `rfm_analysis_plots` module to get the purchase frequency bar chart:

In [19]:
fig = rfm_analysis_plots.plot_customer_purchase_frequency_bar_chart(df=customer_order_frequency_interpretation_counts)

<a id='frequency-analysis-summary'></a>
## Summary:

### Call the `frequency_analysis_summary()` function from `rfm_analysis_summary` <br> <br> module to summarize the `Frequency Analysis` section:

In [20]:
rfm_analysis_summary.get_frequency_analysis_summary(customer_order_frequency_interpretation_counts, fig)


## Frequency Analysis Conclusion:

Frequency measures **how often a customer makes a complete purchase**.

In this analysis:
- Only orders with `delivered` status were considered to ensure that each counted order represents a 
**completed and revenue-realized transaction**.
- Since a single customer in the OLIST dataset may have multiple `customer_id`s across different orders, 
all frequency calculations were performed at the **`customer_unique_id` level**, which represents a real-world customer.
- The frequency metric was calculated as the **number of distinct orders per customer**.

The resulting table `customer_order_frequency_interpretation` contains **one row per unique customer**, along with:
- `total_orders`: total number of completed purchases
- `Interpretation`: a business-friendly customer segment based on purchase behavior.

To make the results interpretable from a business perspective, customers were segmented as follows:

- **One-Time Buyer**: 1 order  
- **Returning Customer**: 2 orders  
- **Loyal Customer**: 3 orders  
- **Very Loyal Customer**: 4 orders  
- **VIP Customers**: 5 or more orders


,Interpretation,Total Counts,Customer Share (%)
0,One-Time Buyer,90557,97.000
1,Returning Customer,2573,2.756
2,Loyal Customer,181,0.194
3,Very Loyal,28,0.030
4,VIP Customers,19,0.020



The frequency distribution shows a **strongly right-skewed pattern**, indicating that the majority of 
customers placed only a single order:

- `One-Time Buyer`s dominate the customer base, accounting for **96.947% of all customers**
- Returning and Loyal customers form a much smaller but strategically important segment
- VIP Customers represent a very small fraction but are likely to contribute **disproportionately higher lifetime value**

To properly visualize this imbalance, a **logarithmic scale** was used in the frequency distribution chart. 
This approach prevents one-time buyers from visually overwhelming the chart and allows meaningful comparison 
between smaller yet valuable customer segments.


Overall, the Frequency analysis reveals that the business is highly dependent on **first-time buyers**, highlighting a significant opportunity for:
- Retention strategies
- Repeat purchase incentives
- Loyalty and CRM programs

These insights will be especially valuable when combined with **Monetary value** in the final RFM segmentation.


<a id='monetary-analysis'></a>
## 3.3 Monetary Analysis:

- Monetary asks **"How much money did each customer spend in total??"**
- This answers:
    - Who are the highest-value customers?
    - Who looks loyal but actually spends very little?
    - Who spends a lot but purchases infrequently?


### 3.3.1 Call the `get_total_spending_per_customer()` function from `rfm_analysis_metrics` module to get the information about total spendings per customer:

In [21]:
total_spendings_per_customer = rfm_analysis_metrics.get_total_spending_per_customer(conn=conn,
                                                                                   sql_base_path=SQL_BASE_PATH)

In [22]:
print(f"The `total_spendings_per_customer` table has a total {total_spendings_per_customer.shape[0]} rows and "
      f"{total_spendings_per_customer.shape[1]} features.")

The `total_spendings_per_customer` table has a total 93358 rows and 2 features.


In [23]:
# Below is the information, the table holds:
total_spendings_per_customer.head()

,customer_unique_id,total_amount
0,0a0a92112bd4c708ca5fde585afaa872,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,7571.63
2,763c8b1c9c68a0229c42c9fc6f662b93,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,6929.31
4,459bef486812aa25204be022145caa62,6922.21


### 3.3.2 Merging `customer_order_frequency_interpretation` and `total_spendings_per_customer` dataframes:

 Now, the code below is used to merge these two dataframes in order to get the customer's information about:
 1. Total number of orders
 2. The cusomer's interpreation (i.e. which buying category do the belong to)
 3. The total amount the customer spent so far

In [24]:
total_frequency_and_total_amount = pd.merge(customer_order_frequency_interpretation, 
               total_spendings_per_customer, on='customer_unique_id' , how='inner')

The merged dataframe `total_frequency_and_total_amount` holds the information as displayed below:

In [25]:
total_frequency_and_total_amount.head()

,customer_unique_id,total_orders,Interpretation,total_amount
0,0000366f3b9a7992bf8c76cfdf3221e2,1,One-Time Buyer,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,One-Time Buyer,27.19
2,0000f46a3911fa3c0805444483337064,1,One-Time Buyer,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,One-Time Buyer,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,One-Time Buyer,196.89


### 3.3.3 Grouping the customers based on `Interpretation` feature:

In [26]:
segment_monetary_summary = (total_frequency_and_total_amount.groupby('Interpretation')
                            .agg(total_customers=('customer_unique_id', 'nunique'),
                            total_revenue=('total_amount', 'sum'),
                            avg_revenue_per_customer=('total_amount', 'mean'))
                            .reset_index()
                            .sort_values(by='total_revenue', ascending=False))

The dataframe was grouped based on `Interpretation` feature to group them into those 5 bins to analyze the customers behavior
based on their buying frequency:

In [27]:
segment_monetary_summary

,Interpretation,total_customers,total_revenue,avg_revenue_per_customer
1,One-Time Buyer,90557,14555586.29,160.733972
2,Returning Customer,2573,748811.57,291.026650
0,Loyal Customer,181,78333.95,432.784254
4,Very Loyal,28,22086.25,788.794643
3,VIP Customers,19,14955.69,787.141579


### 3.3.4 Merging `segment_monetary_summary` & `customer_order_frequency_interpretation_counts` to get the full table for visual comparison:

In [28]:
segment_monetary_summary_w_customer_share = pd.merge(segment_monetary_summary,
                                                    customer_order_frequency_interpretation_counts,
                                                     on='Interpretation',
                                                    how='inner').drop(columns=['Total Counts'])

segment_monetary_summary_w_customer_share = (segment_monetary_summary_w_customer_share
                                             .rename(columns={'total_customers':'Total Customers',
                                                             'total_revenue': 'Total Revenue',
                                                             'avg_revenue_per_customer': 'Average Revenue Per Customer',
                                                             'Customer Share (%)':'Revenue Contribution (%)'}))

The resulted merged table `segment_monetary_summary_w_customer_share` will be used to compare the customers 
using a visualization, where the key idea would be to compare  **`VOLUME`** **V/S**  **`VALUE`**, the table `segment_monetary_summary_w_customer_share` is displayed below:

In [29]:
segment_monetary_summary_w_customer_share

,Interpretation,Total Customers,Total Revenue,Average Revenue Per Customer,Revenue Contribution (%)
0,One-Time Buyer,90557,14555586.29,160.733972,97.000
1,Returning Customer,2573,748811.57,291.026650,2.756
2,Loyal Customer,181,78333.95,432.784254,0.194
3,Very Loyal,28,22086.25,788.794643,0.030
4,VIP Customers,19,14955.69,787.141579,0.020


### 3.3.5 Call the `plot_volume_vs_value_chart()` function from `rfm_analysis_plots` module to plot the VOLUME V/S VALUE chart:

In [30]:
fig = rfm_analysis_plots.plot_volume_vs_value_chart(df=segment_monetary_summary_w_customer_share)

In [31]:
reload_package()

Packages reloaded!!


<a id='monetary-analysis-summary'></a>
## Summary:

### Call the `get_monetary_analysis_summary()` function from `rfm_analysis_summary` <br> <br> module to summarize the Monetary Analysis section:

In [32]:
rfm_analysis_summary.get_monetary_analysis_summary(segment_monetary_summary_w_customer_share,
                                                            fig)


## Monetary Analysis Conclusion:

Monetary analysis focuses on understanding **how much revenue customers generate**, not just how often they purchase.
While Frequency identifies repeat behavior, **Monetary highlights customer value**.

In this analysis:
- Only **delivered orders** were considered to ensure that revenue reflects completed transactions.
- Monetary value was calculated at the **order item level** to capture the true transaction amount, then aggregated to the
  `customer_unique_id` level.
- The monetary data was joined with the previously defined **Frequency-based customer segments**, allowing revenue
  contribution and customer value to be analyzed together.

,Interpretation,Total Customers,Total Revenue,Average Revenue Per Customer,Revenue Contribution (%)
0,One-Time Buyer,90557,14555586.29,160.733972,97.000
1,Returning Customer,2573,748811.57,291.026650,2.756
2,Loyal Customer,181,78333.95,432.784254,0.194
3,Very Loyal,28,22086.25,788.794643,0.030
4,VIP Customers,19,14955.69,787.141579,0.020


The chart below displays the comparison between the <b>Volume</b> and <b>Value</b>

<a id='key-findings--business-interpretation'></a>
# 4. Key Findings & Business Interpretation:

In [33]:
reload_package()

Packages reloaded!!


In [34]:
rfm_analysis_summary.get_key_findings_and_intrerp()


### Key Findings:

- **One-Time Buyers** generate the **highest total revenue** overall, primarily due to their overwhelming volume
  (≈97% of the customer base), despite having the **lowest average revenue per customer**.
- **Returning, Loyal, and Very Loyal customers**, although significantly fewer in number, contribute **progressively higher
  average revenue per customer**, indicating stronger individual customer value.
- **VIP Customers** represent the smallest segment but show **one of the highest average revenues per customer**,
  highlighting their importance from a long-term value perspective rather than volume.
- The contrast between **total revenue (volume-driven)** and **average revenue per customer (value-driven)** reveals that
  revenue concentration alone can be misleading without considering customer quality.

### Business Interpretation:

- Revenue is currently **volume-driven**, dominated by One-Time Buyers.
- However, **high-frequency segments deliver disproportionately higher value per customer**, making them ideal targets
  for:
  - Retention strategies
  - Personalized marketing
  - Loyalty and reward programs

The final visualization effectively demonstrates this dual perspective by:
- Using **Log₁₀ scaled total revenue** to highlight revenue concentration.
- Overlaying **average revenue per customer** to reveal customer value across segments.

This analysis reinforces the importance of balancing **customer acquisition** with **customer retention and value growth**
to build a sustainable revenue strategy.
